In [11]:
!pip install evaluate
!pip install tiktoken

In [12]:
from datasets import Dataset
import evaluate
import os
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification, AutoTokenizer
import pandas as pd
import torch
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
columns = ["question_translated", "context", "answerable", "lang"]
train_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/NLP/train_translated.csv")
val_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/NLP/validation_translated.csv")
train_df = train_df[columns]
val_df   = val_df[columns]


train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)


In [14]:
model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess(examples):
    texts = [q + " [SEP] " + c for q, c in zip(examples["question_translated"], examples["context"])]
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokenized["labels"] = [int(a) for a in examples["answerable"]]

    tokenized["lang"] = examples["lang"]
    return tokenized


train_dataset = train_dataset.map(preprocess, batched=True)
val_dataset   = val_dataset.map(preprocess, batched=True)


train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels", "lang"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels", "lang"])


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/6335 [00:00<?, ? examples/s]

Map:   0%|          | 0/1155 [00:00<?, ? examples/s]

In [15]:
languages = ["te"]

train_by_lang = {lang: train_dataset.filter(lambda x: x["lang"].lower() == lang) for lang in languages}
val_by_lang   = {lang: val_dataset.filter(lambda x: x["lang"].lower() == lang) for lang in languages}


Filter:   0%|          | 0/6335 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1155 [00:00<?, ? examples/s]

In [16]:
columns_to_keep = ["input_ids", "attention_mask", "labels"]

for lang in languages:
    train_by_lang[lang] = train_by_lang[lang].remove_columns(
        [c for c in train_by_lang[lang].column_names if c not in columns_to_keep]
    )
    val_by_lang[lang] = val_by_lang[lang].remove_columns(
        [c for c in val_by_lang[lang].column_names if c not in columns_to_keep]
    )

    train_by_lang[lang].set_format(type="torch", columns=columns_to_keep)
    val_by_lang[lang].set_format(type="torch", columns=columns_to_keep)


In [17]:
models_by_lang = {
    lang: AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )
    for lang in languages
}

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
for lang in languages:
    train_dataset = train_by_lang[lang]
    print(f"Training for lang: {lang}")
    training_args = TrainingArguments(
        output_dir=f'./results_{lang}',
        per_device_train_batch_size=6,
        num_train_epochs=5,
        learning_rate=2e-5,
        save_total_limit=1,
        logging_steps=50,
        report_to=[]
    )

    trainer = Trainer(
        model=models_by_lang[lang],
        args=training_args,
        train_dataset=train_dataset,
        tokenizer=tokenizer
    )

    trainer.train()


/tmp/ipython-input-2948409813.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Training for lang: te


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Step,Training Loss
50,0.271900
100,0.187800
150,0.114800
200,0.079000
250,0.063400
300,0.157700
350,0.098800
400,0.124900
450,0.033700
500,0.059700


In [19]:
metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    accuracy = metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    return {"accuracy": accuracy["accuracy"], "f1": f1["f1"]}

for lang in languages:
    print(f"Evaluating model for language: {lang}")

    model = models_by_lang[lang]
    val_dataset_lang = val_by_lang[lang]

    val_dataset_lang.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"]
    )

    eval_args = TrainingArguments(
        output_dir=f"./eval_{lang}",
        per_device_eval_batch_size=8,
        do_train=False,
        do_eval=True,
        logging_dir=f"./logs_{lang}",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=eval_args,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    eval_result = trainer.evaluate(eval_dataset=val_dataset_lang)
    print(f"Validation results for {lang}: {eval_result}")

Evaluating model for language: te


/tmp/ipython-input-2400240166.py:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Validation results for te: {'eval_loss': 0.7819141745567322, 'eval_model_preparation_time': 0.0027, 'eval_accuracy': 0.859375, 'eval_f1': 0.9137380191693291, 'eval_runtime': 18.097, 'eval_samples_per_second': 21.219, 'eval_steps_per_second': 2.652}


## **Test data (week 41+)**

In [25]:
columns = ["question", "context", "answerable", "lang"]
val_df = pd.read_json("/content/drive/MyDrive/Colab Notebooks/NLP/test-only-eng.json")
val_df   = val_df[columns]

val_dataset   = Dataset.from_pandas(val_df)

In [26]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess(examples):
    texts = [q + " [SEP] " + c for q, c in zip(examples["question"], examples["context"])]
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokenized["labels"] = [int(a) for a in examples["answerable"]]

    tokenized["lang"] = examples["lang"]
    return tokenized


val_dataset   = val_dataset.map(preprocess, batched=True)

val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels", "lang"])

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

In [27]:
languages = ["en"]

val_by_lang   = {lang: val_dataset.filter(lambda x: x["lang"].lower() == lang) for lang in languages}

Filter:   0%|          | 0/32 [00:00<?, ? examples/s]

In [29]:
metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    accuracy = metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    return {"accuracy": accuracy["accuracy"], "f1": f1["f1"]}

for lang in languages:
    print(f"Evaluating model for language: {lang}")

    model = models_by_lang['te']
    val_dataset_lang = val_by_lang[lang]

    val_dataset_lang.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"]
    )

    eval_args = TrainingArguments(
        output_dir=f"./eval_{lang}",
        per_device_eval_batch_size=8,
        do_train=False,
        do_eval=True,
        logging_dir=f"./logs_{lang}",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=eval_args,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    eval_result = trainer.evaluate(eval_dataset=val_dataset_lang)
    print(f"Validation results for {lang}: {eval_result}")

Evaluating model for language: en


/tmp/ipython-input-2801065017.py:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Validation results for en: {'eval_loss': 2.6475772857666016, 'eval_model_preparation_time': 0.003, 'eval_accuracy': 0.6875, 'eval_f1': 0.8148148148148148, 'eval_runtime': 1.5425, 'eval_samples_per_second': 20.746, 'eval_steps_per_second': 2.593}
